# 01. Предобработка и генерация признаков для Paddy Dataset

## Цель ноутбука

В этом ноутбуке подготовим данные о выращивании риса для последующего машинного обучения. Датасет содержит сельскохозяйственные, почвенные, погодные и технологические показатели, а целевая переменная — `Paddy yield(in Kg)`, урожайность риса в килограммах.

Основные этапы:

1. первичный аудит и очистка данных;
2. проверка пропусков и дубликатов;
3. первичный анализ с графиками;
4. разделение данных на train/test **до обучения преобразований**;
5. генерация новых признаков;
6. обработка числовых и категориальных признаков через `ColumnTransformer`;
7. кодирование категориальных признаков;
8. Standard Scaling и Min-Max Scaling;
9. демонстрация Label Encoding и Target Encoding как альтернатив;
10. сохранение готового pipeline и обработанных матриц.

> **Главный принцип воспроизводимости:** все преобразования, которые должны применяться к новым данным, собираются в `Pipeline`/`ColumnTransformer`. Они обучаются только на тренировочной части данных. Это помогает избежать data leakage — ситуации, когда информация из тестовой выборки случайно попадает в процесс обучения преобразований. `Pipeline` предназначен именно для последовательного применения преобразователей, а `ColumnTransformer` позволяет выполнять разные преобразования для разных групп столбцов.

## 1. Откуда взялись данные

набор содержит 45 столбцов, включая площадь поля, агроблок, сорт риса, тип почвы, нормы внесения удобрений, показатели дренажа/орошения, температуру, ветер, влажность и целевую урожайность. Целевая переменная — `Paddy yield(in Kg)`.

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    OrdinalEncoder,
    StandardScaler,
    MinMaxScaler,
    TargetEncoder,
)
import joblib

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATA_PATHS = [
    Path("data/paddydataset.csv"),
    Path("../data/paddydataset.csv"),
    Path("paddydataset.csv"),
]

DATA_PATH = next((p for p in DATA_PATHS if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Не найден paddydataset.csv. Положите файл в папку data/ или рядом с ноутбуком."
    )

print(f"Используем файл: {DATA_PATH.resolve()}")

## 2. Загрузка и первичный аудит

Сначала ничего не исправляем «вслепую». Нам важно понять:

- сколько строк и столбцов;
- какие типы данных определил pandas;
- сколько пропусков;
- сколько уникальных значений в категориальных столбцах;
- есть ли полные дубликаты.

Такой аудит позволяет отделить реальную проблему данных от нормального поведения набора. Например, отсутствие пропусков означает, что агрессивное заполнение `NaN` сейчас не требуется, но `SimpleImputer` всё равно можно оставить внутри pipeline как защитный механизм для будущих данных.

In [ ]:
df_raw = pd.read_csv(DATA_PATH)
# Убираем случайные пробелы в названиях столбцов. В исходном CSV у Hectares есть завершающий пробел.
df_raw.columns = df_raw.columns.str.strip()
df = df_raw.copy()

audit = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "missing_%": (df.isna().mean() * 100).round(2),
    "unique": df.nunique(dropna=True),
})

print(f"Размер данных: {df.shape[0]:,} строк × {df.shape[1]} столбцов")
print(f"Полных дубликатов: {df.duplicated().sum():,}")
print(f"Всего пропусков: {int(df.isna().sum().sum()):,}")

display(audit.sort_values(["missing", "unique"], ascending=[False, True]))

### Что показывает аудит

В этом CSV пропусков нет, поэтому удалять строки из-за `NaN` не нужно. Однако обнаруживаются полные дубликаты. Поскольку строки совпадают **по всем признакам и целевой переменной**, удаление точных дубликатов является безопасной детерминированной операцией для подготовки этого набора: мы не теряем уникальную информацию.

Важно отличать точный дубликат от просто похожего наблюдения. Похожие строки удалять автоматически нельзя — они могут описывать разные поля или разные хозяйственные условия.

In [ ]:
# Количество пропусков по столбцам
missing = df.isna().sum().sort_values(ascending=False)
print("Столбцы с пропусками:")
display(missing[missing > 0].to_frame("missing"))

# Точные дубликаты
duplicate_count = int(df.duplicated().sum())
print(f"Точных дубликатов: {duplicate_count:,}")

## 3. Очистка точных дубликатов

Удаляем только полные дубликаты. Это не является обучаемым преобразованием, поэтому операция выполняется **до train/test split** и явно фиксируется в коде.

Пропуски не заполняем вручную, потому что в текущем файле их нет. Для воспроизводимого pipeline ниже всё равно добавим `SimpleImputer`: если в будущем появится пропуск в числовом признаке, будет использована медиана, а для категориального — наиболее частое значение. Это делает pipeline устойчивее к новым данным.

In [ ]:
df_clean = df.drop_duplicates().reset_index(drop=True)
print(f"Было строк: {len(df):,}")
print(f"После удаления точных дубликатов: {len(df_clean):,}")
print(f"Удалено: {len(df) - len(df_clean):,}")

## 4. Первичный анализ распределений

Графики здесь нужны не для построения итогового отчёта, а для контроля качества данных.

### График 1 — распределение урожайности

Гистограмма показывает, сколько наблюдений приходится на разные диапазоны урожайности. Она помогает увидеть форму распределения, асимметрию и потенциально необычные значения.

### График 2 — урожайность по сортам

Boxplot показывает медиану, центральные 50% наблюдений и возможные крайние значения урожайности для каждого сорта. Это полезно для проверки того, отличается ли распределение целевой переменной между категориями.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].hist(df_clean["Paddy yield(in Kg)"], bins=30)
axes[0].set_title("Распределение урожайности риса")
axes[0].set_xlabel("Paddy yield, кг")
axes[0].set_ylabel("Количество наблюдений")

df_clean.boxplot(column="Paddy yield(in Kg)", by="Variety", ax=axes[1])
axes[1].set_title("Распределение урожайности по сортам")
axes[1].set_xlabel("Сорт риса")
axes[1].set_ylabel("Paddy yield, кг")

plt.tight_layout()
plt.show()

## 5. Проверка числовых признаков и выбросов

Для числовых признаков посмотрим на описательную статистику и boxplot. В этом наборе многие агрономические показатели принимают ограниченное число дискретных значений. Поэтому правило «всё, что далеко от среднего, удалить» здесь было бы неправильным.

**Решение:** автоматическое удаление/обрезание выбросов не выполняем. Значение может быть редким, но при этом физически корректным. Вместо этого масштабирование будет выполнено внутри pipeline, а устойчивое к выбросам `RobustScaler` можно выбрать позже при сравнении моделей.

In [ ]:
numeric_cols_raw = df_clean.select_dtypes(include=np.number).columns.tolist()
display(df_clean[numeric_cols_raw].describe().T)

# Покажем boxplot для нескольких ключевых признаков
plot_cols = [
    "Hectares",
    "Seedrate(in Kg)",
    "Trash(in bundles)",
    "Paddy yield(in Kg)",
]

fig, axes = plt.subplots(1, len(plot_cols), figsize=(16, 4))
for ax, col in zip(axes, plot_cols):
    ax.boxplot(df_clean[col].dropna(), vert=True)
    ax.set_title(col)
    ax.set_ylabel("")
plt.tight_layout()
plt.show()

## 6. Проверка категориальных признаков

Категориальные признаки нельзя напрямую передавать большинству моделей в виде строк. Сначала их нужно преобразовать в числа.

Для этого датасета категориальными являются:

- `Agriblock` — сельскохозяйственный блок;
- `Variety` — сорт;
- `Soil Types` — тип почвы;
- `Nursery` — тип питомника;
- четыре направления ветра для разных периодов.

One-Hot Encoding создаёт отдельный бинарный признак для каждой категории. Например, `Soil Types = clay/alluvial` превращается в отдельные числовые столбцы. `OneHotEncoder` поддерживает обработку неизвестных категорий через `handle_unknown="ignore"`, что особенно важно для новых данных. citeturn0search0turn0search11

In [ ]:
categorical_cols_raw = df_clean.select_dtypes(include=["object", "category"]).columns.tolist()

cat_summary = pd.DataFrame({
    "unique_values": df_clean[categorical_cols_raw].nunique(),
    "examples": [", ".join(map(str, df_clean[c].dropna().unique()[:8])) for c in categorical_cols_raw],
})
display(cat_summary)

## 7. Корреляции: только как диагностический инструмент

Корреляция помогает увидеть линейную связь между числовыми признаками и урожайностью. Но корреляция не означает причинность.

Кроме того, в этом наборе часть признаков естественно связана с размером поля (`Hectares`). Поэтому ниже мы не удаляем признаки только на основании корреляции: окончательный выбор признаков должен зависеть от задачи и выбранной модели.

In [ ]:
num_corr = df_clean.select_dtypes(include=np.number).corr()

# Берём 15 числовых признаков с самой сильной по модулю корреляцией с target
corr_to_target = num_corr["Paddy yield(in Kg)"].drop("Paddy yield(in Kg)")
top_corr_cols = corr_to_target.abs().sort_values(ascending=False).head(15).index.tolist()

plt.figure(figsize=(10, 8))
sns.heatmap(
    df_clean[top_corr_cols + ["Paddy yield(in Kg)"]].corr(),
    cmap="coolwarm",
    center=0,
    annot=False,
)
plt.title("Корреляции наиболее связанных с урожайностью числовых признаков")
plt.tight_layout()
plt.show()

# 8. Разделение признаков и целевой переменной

Целевая переменная `Paddy yield(in Kg)` не должна проходить через преобразование признаков вместе с `X`.

Разделяем данные так:

- `X` — все признаки, по которым будем делать прогноз;
- `y` — урожайность, которую хотим прогнозировать.

Затем делим `X` и `y` на train/test. Только после этого обучаем imputers, encoders и scalers. Это ключевой момент для предотвращения утечки информации из теста.

In [ ]:
TARGET = "Paddy yield(in Kg)"

X = df_clean.drop(columns=[TARGET]).copy()
y = df_clean[TARGET].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test:  {y_test.shape}")

# 9. Генерация новых признаков

Сырые показатели не всегда являются самым удобным представлением для модели. Поэтому создадим признаки, которые выражают более понятные отношения.

### Что генерируем

**Интенсивность ресурсов:**

- `seedrate_per_ha` — семена на гектар;
- `nursery_area_per_ha` — площадь питомника на гектар;
- `trash_per_ha` — количество растительных остатков на гектар.

**Водный баланс:**

- `total_drain_mm` — суммарный дренаж;
- `total_irrigation_mm` — суммарный показатель DAI/AI;
- `water_balance_mm` — разница между орошением и дренажом.

**Погодные агрегаты:**

- средняя минимальная температура;
- средняя максимальная температура;
- средняя температура и средний температурный диапазон;
- средняя скорость ветра;
- средняя влажность;
- диапазон влажности.

Такие признаки не используют целевую переменную, поэтому их можно безопасно строить одинаковым способом для train и test. `FunctionTransformer` также может использоваться для stateless-преобразований, но здесь собственный sklearn-compatible transformer удобнее, потому что он явно работает с именами столбцов. citeturn0search6

In [ ]:
class PaddyFeatureGenerator(BaseEstimator, TransformerMixin):
    """Генерация предметно-ориентированных признаков для Paddy Dataset."""

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        eps = 1e-9

        # Интенсивность на единицу площади
        X["seedrate_per_ha"] = X["Seedrate(in Kg)"] / (X["Hectares"] + eps)
        X["nursery_area_per_ha"] = X["Nursery area (Cents)"] / (X["Hectares"] + eps)
        X["trash_per_ha"] = X["Trash(in bundles)"] / (X["Hectares"] + eps)

        # Водный режим
        drain_cols = [
            "30DRain( in mm)",
            "30_50DRain( in mm)",
            "51_70DRain(in mm)",
            "71_105DRain(in mm)",
        ]
        irrigation_cols = [
            "30DAI(in mm)",
            "30_50DAI(in mm)",
            "51_70AI(in mm)",
            "71_105DAI(in mm)",
        ]
        X["total_drain_mm"] = X[drain_cols].sum(axis=1)
        X["total_irrigation_mm"] = X[irrigation_cols].sum(axis=1)
        X["water_balance_mm"] = X["total_irrigation_mm"] - X["total_drain_mm"]

        # Температурные показатели по четырём периодам
        min_temp_cols = [
            "Min temp_D1_D30", "Min temp_D31_D60",
            "Min temp_D61_D90", "Min temp_D91_D120",
        ]
        max_temp_cols = [
            "Max temp_D1_D30", "Max temp_D31_D60",
            "Max temp_D61_D90", "Max temp_D91_D120",
        ]
        X["avg_min_temp"] = X[min_temp_cols].mean(axis=1)
        X["avg_max_temp"] = X[max_temp_cols].mean(axis=1)
        X["avg_temp_range"] = X["avg_max_temp"] - X["avg_min_temp"]

        # Ветер
        wind_cols = [
            "Inst Wind Speed_D1_D30(in Knots)",
            "Inst Wind Speed_D31_D60(in Knots)",
            "Inst Wind Speed_D61_D90(in Knots)",
            "Inst Wind Speed_D91_D120(in Knots)",
        ]
        X["avg_wind_speed"] = X[wind_cols].mean(axis=1)
        X["wind_speed_range"] = X[wind_cols].max(axis=1) - X[wind_cols].min(axis=1)

        # Влажность
        humidity_cols = [
            "Relative Humidity_D1_D30",
            "Relative Humidity_D31_D60",
            "Relative Humidity_D61_D90",
            "Relative Humidity_D91_D120",
        ]
        X["avg_humidity"] = X[humidity_cols].mean(axis=1)
        X["humidity_range"] = X[humidity_cols].max(axis=1) - X[humidity_cols].min(axis=1)

        return X

feature_generator = PaddyFeatureGenerator()
X_train_features = feature_generator.fit_transform(X_train)

new_features = [c for c in X_train_features.columns if c not in X_train.columns]
print(f"Добавлено новых признаков: {len(new_features)}")
print(new_features)
display(X_train_features[new_features].head())

## 10. Формируем списки числовых и категориальных признаков после feature engineering

После генерации новых признаков набор снова делится на две группы:

- **числовые** — пропуск → медиана → масштабирование;
- **категориальные** — пропуск → наиболее частая категория → One-Hot Encoding.

`ColumnTransformer` применит эти две ветки параллельно и объединит их в единую матрицу признаков. Такой подход особенно удобен для смешанных таблиц. citeturn0search2turn0search8

In [ ]:
NUMERIC_FEATURES = X_train_features.select_dtypes(include=np.number).columns.tolist()
CATEGORICAL_FEATURES = X_train_features.select_dtypes(include=["object", "category"]).columns.tolist()

print(f"Числовых признаков: {len(NUMERIC_FEATURES)}")
print(f"Категориальных признаков: {len(CATEGORICAL_FEATURES)}")
print("Категориальные:", CATEGORICAL_FEATURES)

# 11. Основной воспроизводимый pipeline: One-Hot + StandardScaler

### Числовая ветка

`SimpleImputer(strategy="median")` → если появится пропуск, он заменяется медианой тренировочной выборки.

`StandardScaler()` → преобразует признак так, чтобы на обучающей выборке он имел среднее около 0 и стандартное отклонение около 1.

### Категориальная ветка

`SimpleImputer(strategy="most_frequent")` → пропуск заменяется самой частой категорией.

`OneHotEncoder(handle_unknown="ignore")` → категории превращаются в 0/1-признаки. Неизвестная категория в новых данных не ломает pipeline.

Такой pipeline можно сохранить и затем применить к новым данным без ручного повторения шагов. `Pipeline` последовательно обучает и применяет трансформеры, а параметры можно совместно оптимизировать при кросс-валидации. citeturn0search3

In [ ]:
# sparse_output=False делает результат обычным numpy-массивом,
# что удобно для демонстрации и последующего сохранения CSV.
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, NUMERIC_FEATURES),
        ("cat", categorical_pipeline, CATEGORICAL_FEATURES),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

full_pipeline = Pipeline(steps=[
    ("feature_generation", PaddyFeatureGenerator()),
    ("preprocessing", preprocessor),
])

X_train_processed = full_pipeline.fit_transform(X_train, y_train)
X_test_processed = full_pipeline.transform(X_test)

print("Размер после преобразования:")
print("X_train_processed:", X_train_processed.shape)
print("X_test_processed: ", X_test_processed.shape)

In [ ]:
# Имена признаков после преобразования
feature_names = full_pipeline.named_steps["preprocessing"].get_feature_names_out()

print(f"Всего признаков после encoding + scaling: {len(feature_names)}")
display(pd.DataFrame({"feature": feature_names}).head(40))

## 12. Как изменился один числовой признак после Standard Scaling

Например, `Hectares` измеряется в гектарах, а другие признаки — в килограммах, миллиметрах, градусах, узлах и процентах. Эти величины имеют совершенно разные масштабы.

Standard Scaling переводит их в сопоставимую шкалу. Формула:

\[
z = 
rac{x - \mu}{\sigma}
\]

где `μ` — среднее тренировочной выборки, а `σ` — её стандартное отклонение.

Это особенно полезно для моделей, чувствительных к масштабу признаков, например KNN, SVM и некоторых линейных моделей. `StandardScaler` является стандартным инструментом sklearn для стандартизации. citeturn0search1

In [ ]:
processed_df_preview = pd.DataFrame(X_train_processed, columns=feature_names, index=X_train.index)

cols_to_show = [c for c in ["Hectares", "Seedrate(in Kg)", "avg_humidity", "seedrate_per_ha"] if c in processed_df_preview.columns]
display(processed_df_preview[cols_to_show].head())

print("Средние значения выбранных стандартизированных признаков:")
display(processed_df_preview[cols_to_show].mean().round(6).to_frame("mean"))
print("Стандартные отклонения:")
display(processed_df_preview[cols_to_show].std(ddof=0).round(6).to_frame("std"))

# 13. Альтернатива: Min-Max Scaling

Иногда вместо Standard Scaling нужно привести числовые признаки к фиксированному диапазону, обычно `[0, 1]`.

Формула:

\[
x' = 
rac{x - x_{min}}{x_{max} - x_{min}}
\]

Ниже строим **тот же pipeline**, но вместо `StandardScaler` используем `MinMaxScaler`. Это демонстрирует, что масштабирование можно заменить одной строкой внутри воспроизводимой схемы, не переписывая весь preprocessing.

In [ ]:
numeric_minmax_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", MinMaxScaler()),
])

preprocessor_minmax = ColumnTransformer(
    transformers=[
        ("num", numeric_minmax_pipeline, NUMERIC_FEATURES),
        ("cat", categorical_pipeline, CATEGORICAL_FEATURES),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

minmax_pipeline = Pipeline(steps=[
    ("feature_generation", PaddyFeatureGenerator()),
    ("preprocessing", preprocessor_minmax),
])

X_train_minmax = minmax_pipeline.fit_transform(X_train, y_train)
X_test_minmax = minmax_pipeline.transform(X_test)

minmax_names = minmax_pipeline.named_steps["preprocessing"].get_feature_names_out()
minmax_preview = pd.DataFrame(X_train_minmax, columns=minmax_names)

print("Размер:", X_train_minmax.shape)
print("Минимум числовой части:", np.nanmin(X_train_minmax[:, :len(NUMERIC_FEATURES)], axis=0).min().round(6))
print("Максимум числовой части:", np.nanmax(X_train_minmax[:, :len(NUMERIC_FEATURES)], axis=0).max().round(6))

# 14. Label Encoding: что это и почему не используем его как основной метод

**Label Encoding** заменяет категории числами. Например:

- `clay → 0`
- `alluvial → 1`

Проблема: модель может интерпретировать эти числа как порядок и расстояние. Для `clay=0` и `alluvial=1` это ещё не так заметно, но для десятков категорий искусственный порядок становится особенно опасным.

Поэтому для **признаков** здесь лучше использовать One-Hot Encoding. В sklearn для категориальных признаков с порядком можно использовать `OrdinalEncoder`, а `LabelEncoder` традиционно предназначен прежде всего для кодирования целевых меток.

Ниже — демонстрация label-подобного кодирования через `OrdinalEncoder`, чтобы было видно сам принцип.

In [ ]:
label_like_encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1,
)

label_demo = label_like_encoder.fit_transform(df_clean[["Soil Types", "Nursery"]])
label_demo_df = pd.DataFrame(
    label_demo,
    columns=["Soil Types_encoded", "Nursery_encoded"],
)

display(pd.concat([df_clean[["Soil Types", "Nursery"]].reset_index(drop=True).head(10), label_demo_df.head(10)], axis=1))

# 15. Target Encoding — альтернативный вариант для категорий

Target Encoding заменяет категорию числом, связанным со средним значением целевой переменной для этой категории.

Например, для `Variety = CO_43` можно получить среднюю урожайность этого сорта в тренировочной выборке. Это может быть компактнее One-Hot Encoding, особенно если категорий очень много.

Но здесь есть важный риск: если вычислять среднее по **всему датасету**, целевая переменная попадёт обратно в признаки. Это называется target leakage.

Поэтому `TargetEncoder` должен находиться **внутри pipeline** и обучаться только на `X_train, y_train`. В современных версиях sklearn `TargetEncoder` поддерживает внутреннее кросс-валидируемое обучение для снижения leakage. В этом ноутбуке он показан как альтернативный pipeline, а не как безусловная замена One-Hot Encoding.

In [ ]:
target_cat_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    (
        "target_encoder",
        TargetEncoder(
            target_type="continuous",
            smooth="auto",
            cv=5,
            shuffle=True,
            random_state=RANDOM_STATE,
        ),
    ),
])

target_preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, NUMERIC_FEATURES),
        ("cat_target", target_cat_pipeline, CATEGORICAL_FEATURES),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

target_encoding_pipeline = Pipeline(steps=[
    ("feature_generation", PaddyFeatureGenerator()),
    ("preprocessing", target_preprocessor),
])

X_train_target_encoded = target_encoding_pipeline.fit_transform(X_train, y_train)
X_test_target_encoded = target_encoding_pipeline.transform(X_test)

target_names = target_encoding_pipeline.named_steps["preprocessing"].get_feature_names_out()

print("Размер после Target Encoding:", X_train_target_encoded.shape)
display(pd.DataFrame(X_train_target_encoded, columns=target_names).head())

## 16. Сравнение трёх подходов к кодированию

| Метод | Идея | Плюсы | Минусы |
|---|---|---|---|
| One-Hot | отдельный 0/1-признак для категории | просто, понятно, безопасно | увеличивает число столбцов |
| Label/Ordinal | категория → целое число | очень компактно | может создать искусственный порядок |
| Target Encoding | категория → информация о target | компактно, часто полезно при высокой кардинальности | риск leakage, поэтому нужен аккуратный pipeline |

Для данного датасета категорий мало (`Agriblock` — 6, `Variety` — 3 и т.д.), поэтому **One-Hot Encoding является наиболее прозрачным базовым выбором**. UCI также описывает часть переменных как категориальные. citeturn1search0

# 17. Проверка воспроизводимости

Воспроизводимость означает, что при одинаковых входных данных и одинаковом `random_state` мы получаем одинаковое преобразование.

Проверим это практически: создадим второй pipeline с теми же настройками, обучим его на тех же данных и сравним результаты через `np.allclose`.

In [ ]:
# Второй экземпляр полностью той же схемы
full_pipeline_2 = Pipeline(steps=[
    ("feature_generation", PaddyFeatureGenerator()),
    ("preprocessing", ColumnTransformer(
        transformers=[
            ("num", Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]), NUMERIC_FEATURES),
            ("cat", Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
            ]), CATEGORICAL_FEATURES),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )),
])

X_train_processed_2 = full_pipeline_2.fit_transform(X_train, y_train)

print("Одинаковая форма:", X_train_processed.shape == X_train_processed_2.shape)
print("Одинаковые значения:", np.allclose(X_train_processed, X_train_processed_2))

# 18. Сохранение pipeline и готовых признаков

Сохраняем:

1. `paddy_preprocessing_pipeline.joblib` — обученный основной preprocessing pipeline;
2. `paddy_feature_names.csv` — названия признаков после преобразования;
3. `X_train_processed.csv` и `X_test_processed.csv` — готовые матрицы признаков;
4. `y_train.csv` и `y_test.csv` — соответствующие значения target.

После сохранения pipeline новые данные можно преобразовывать теми же правилами через `.transform()`, не вычисляя медианы, категории и параметры масштабирования заново.

In [ ]:
ARTIFACTS_DIR = Path("artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(full_pipeline, ARTIFACTS_DIR / "paddy_preprocessing_pipeline.joblib")
pd.DataFrame({"feature": feature_names}).to_csv(
    ARTIFACTS_DIR / "paddy_feature_names.csv", index=False
)
pd.DataFrame(X_train_processed, columns=feature_names, index=X_train.index).to_csv(
    ARTIFACTS_DIR / "X_train_processed.csv", index=False
)
pd.DataFrame(X_test_processed, columns=feature_names, index=X_test.index).to_csv(
    ARTIFACTS_DIR / "X_test_processed.csv", index=False
)
y_train.to_csv(ARTIFACTS_DIR / "y_train.csv", index=False)
y_test.to_csv(ARTIFACTS_DIR / "y_test.csv", index=False)

print("Сохранено:")
for p in sorted(ARTIFACTS_DIR.iterdir()):
    print(" -", p)

# 19. Финальная схема подготовки данных

Получилась следующая воспроизводимая цепочка:

```text
paddydataset.csv
      │
      ├── первичный аудит
      ├── удаление точных дубликатов
      │
      ├── train/test split
      │
      └── Pipeline
             │
             ├── PaddyFeatureGenerator
             │      ├── признаки на гектар
             │      ├── водный баланс
             │      ├── температурные агрегаты
             │      ├── ветер
             │      └── влажность
             │
             └── ColumnTransformer
                    │
                    ├── числовые
                    │     ├── SimpleImputer(median)
                    │     └── StandardScaler
                    │
                    └── категориальные
                          ├── SimpleImputer(most_frequent)
                          └── OneHotEncoder
```

### Итог

- пропуски в текущем CSV отсутствуют;
- точные дубликаты удалены до разделения на train/test;
- автоматическое удаление выбросов не выполняется, потому что редкие значения могут быть валидными агрономическими наблюдениями;
- категориальные признаки кодируются One-Hot в основном pipeline;
- Label/Ordinal Encoding показан как компактная альтернатива, но не выбран основным методом;
- Target Encoding показан отдельным pipeline и выполняется без ручного leakage;
- числовые признаки масштабируются через StandardScaler, а MinMaxScaler показан как альтернативный вариант;
- новые предметно-ориентированные признаки генерируются внутри pipeline;
- обученный preprocessing сохраняется в `joblib`, поэтому его можно повторно применять к новым данным.

Таким образом, следующий этап проекта может использовать **готовый `paddy_preprocessing_pipeline.joblib` напрямую перед моделью регрессии**, не дублируя preprocessing-код.